<a href="https://colab.research.google.com/github/Joeaviator/AIRLINE_DATA/blob/main/AIRLINE_DATA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [36]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier

In [2]:
from google.colab import files
file = files.upload()

Saving Airline Data.csv to Airline Data.csv


In [4]:
df=pd.read_csv("Airline Data.csv")
df.head()

,Unnamed: 0,id,Gender,Customer Type,Age,Type of Travel,Class,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,...,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Inflight service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,satisfaction
0,0,70172,Male,Loyal Customer,13,Personal Travel,Eco Plus,460,3,4,...,5,4,3,4,4,5,5,25,18.0,neutral or dissatisfied
1,1,5047,Male,disloyal Customer,25,Business travel,Business,235,3,2,...,1,1,5,3,1,4,1,1,6.0,neutral or dissatisfied
2,2,110028,Female,Loyal Customer,26,Business travel,Business,1142,2,2,...,5,4,3,4,4,4,5,0,0.0,satisfied
3,3,24026,Female,Loyal Customer,25,Business travel,Business,562,2,5,...,2,2,5,3,1,4,2,11,9.0,neutral or dissatisfied
4,4,119299,Male,Loyal Customer,61,Business travel,Business,214,3,3,...,3,3,4,4,3,3,3,0,0.0,satisfied


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103904 entries, 0 to 103903
Data columns (total 25 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   Unnamed: 0                         103904 non-null  int64  
 1   id                                 103904 non-null  int64  
 2   Gender                             103904 non-null  object 
 3   Customer Type                      103904 non-null  object 
 4   Age                                103904 non-null  int64  
 5   Type of Travel                     103904 non-null  object 
 6   Class                              103904 non-null  object 
 7   Flight Distance                    103904 non-null  int64  
 8   Inflight wifi service              103904 non-null  int64  
 9   Departure/Arrival time convenient  103904 non-null  int64  
 10  Ease of Online booking             103904 non-null  int64  
 11  Gate location                      1039

In [6]:
df.isnull().sum()

,0
Unnamed: 0,0
id,0
Gender,0
Customer Type,0
Age,0
Type of Travel,0
Class,0
Flight Distance,0
Inflight wifi service,0
Departure/Arrival time convenient,0


In [7]:
df.columns

Index(['Unnamed: 0', 'id', 'Gender', 'Customer Type', 'Age', 'Type of Travel',
       'Class', 'Flight Distance', 'Inflight wifi service',
       'Departure/Arrival time convenient', 'Ease of Online booking',
       'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort',
       'Inflight entertainment', 'On-board service', 'Leg room service',
       'Baggage handling', 'Checkin service', 'Inflight service',
       'Cleanliness', 'Departure Delay in Minutes', 'Arrival Delay in Minutes',
       'satisfaction'],
      dtype='object')

In [20]:
df=df.dropna()

In [29]:
df.drop(columns=['Arrival Delay in Minutes'], inplace=True)

In [11]:
label_en=LabelEncoder()

In [12]:
df['Gender']=label_en.fit_transform(df['Gender'])
df['Customer Type']=label_en.fit_transform(df['Customer Type'])
df['Type of Travel']=label_en.fit_transform(df['Type of Travel'])
df['Class']=label_en.fit_transform(df['Class'])
df['satisfaction']=label_en.fit_transform(df['satisfaction'])

In [22]:
df.head()

,Unnamed: 0,Gender,Customer Type,Age,Type of Travel,Class,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,...,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Inflight service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,satisfaction
0,0,1,0,13,1,2,460,3,4,3,...,5,4,3,4,4,5,5,25,18.0,0
1,1,1,1,25,0,0,235,3,2,3,...,1,1,5,3,1,4,1,1,6.0,0
2,2,0,0,26,0,0,1142,2,2,2,...,5,4,3,4,4,4,5,0,0.0,1
3,3,0,0,25,0,0,562,2,5,5,...,2,2,5,3,1,4,2,11,9.0,0
4,4,1,0,61,0,0,214,3,3,3,...,3,3,4,4,3,3,3,0,0.0,1


In [23]:
x=df.drop('satisfaction',axis=1)
y=df['satisfaction']

In [24]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [32]:
scale = StandardScaler()

# Drop 'Unnamed: 0' column if it exists. It's often an index and not a feature.
if 'Unnamed: 0' in x_train.columns:
    x_train = x_train.drop('Unnamed: 0', axis=1)
if 'Unnamed: 0' in x_test.columns:
    x_test = x_test.drop('Unnamed: 0', axis=1)

# Process each column to ensure it's numeric and handle NaNs
for col in x_train.columns:
    # If the column is an object type, try to convert strings to numbers
    if x_train[col].dtype == 'object':
        x_train[col] = x_train[col].replace('No Value Avilable', np.nan) # Replace the specific string
        x_train[col] = pd.to_numeric(x_train[col], errors='coerce') # Coerce any other non-numeric to NaN

    # Impute NaNs in the training set
    if x_train[col].isnull().any():
        # Calculate mean excluding NaNs. If all values are NaN, mean will be NaN.
        mean_val_train = x_train[col].dropna().mean()
        # Fill NaNs with the calculated mean, default to 0 if mean is NaN
        x_train[col] = x_train[col].fillna(mean_val_train if pd.notna(mean_val_train) else 0)

for col in x_test.columns:
    # If the column is an object type, try to convert strings to numbers
    if x_test[col].dtype == 'object':
        x_test[col] = x_test[col].replace('No Value Avilable', np.nan)
        x_test[col] = pd.to_numeric(x_test[col], errors='coerce')

    # Impute NaNs in the test set using the mean from the training set to prevent data leakage
    if x_test[col].isnull().any():
        if col in x_train.columns:
            # Use the mean from the already imputed training column
            mean_val_from_train = x_train[col].mean()
            x_test[col] = x_test[col].fillna(mean_val_from_train if pd.notna(mean_val_from_train) else 0)
        else:
            # Fallback for columns not in train, or if train mean is also NaN
            x_test[col] = x_test[col].fillna(0)

# Final explicit conversion to float to ensure all data for StandardScaler is numeric
x_train = x_train.astype(float)
x_test = x_test.astype(float)

# Perform scaling
x_train = scale.fit_transform(x_train)
x_test = scale.transform(x_test)

/tmp/ipykernel_1787/1846313338.py:13: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  x_train[col] = x_train[col].replace('No Value Avilable', np.nan) # Replace the specific string
/tmp/ipykernel_1787/1846313338.py:26: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  x_test[col] = x_test[col].replace('No Value Avilable', np.nan)


In [33]:
Log = LogisticRegression()
Log.fit(x_train, y_train)
preds=Log.predict(x_test)
accuracy_score(y_test, preds)

0.8777729656898128

In [34]:
Knn = KNeighborsClassifier(n_neighbors=3)
Knn.fit(x_train,y_train)
preds=Knn.predict(x_test)
accuracy_score(y_test, preds)

0.9261825706173908

In [37]:
Tree = DecisionTreeClassifier()
Tree.fit(x_train,y_train)
preds=Tree.predict(x_test)
accuracy_score(y_test, preds)

0.9468745488667533